# Employee Data Analysis with PySpark and Spark SQL

Final project of the IBM **Introduction to Big Data with Spark and Hadoop** course (Coursera), extended with cross-checks and verification.

Each task is solved with the requested method and, where useful, **cross-checked with a second method** (Spark SQL vs DataFrame API), so every answer is confirmed by two independent routes.

| | |
|---|---|
| Dataset | `employees.csv`: 50 employees, 5 columns |
| Engine | PySpark (DataFrame API + Spark SQL) |
| Key ideas | lazy evaluation, immutability, explicit schemas, deterministic ordering |

## Setup

In [ ]:
# In the IBM lab: !pip install pyspark findspark wget
import os
from pyspark.sql import SparkSession
# "functions as F" instead of importing max, sum, round directly:
# those names would replace Python's built-in max(), sum() and round().
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

spark = (
    SparkSession.builder
    .appName("Employee Data Analysis")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

# Works from the repository (notebooks/) and from the IBM lab (file in the same folder)
DATA_PATH = "../data/employees.csv" if os.path.exists("../data/employees.csv") else "employees.csv"

### Inspect the file before loading it

`wc -l` counts **newline characters**, not lines. The file has no newline after its last row, so `wc -l` reports 50 for a file with 51 lines (header + 50 employees). Spark's `count()` is the number to trust.

In [ ]:
with open(DATA_PATH, encoding="utf-8") as fh:
    for _ in range(5):
        print(fh.readline().rstrip())

## Task 1: Generate a DataFrame from the CSV data

`inferSchema=True` lets Spark guess the types. Convenient, but Spark must read the data an extra time to guess, and the guess can be wrong.

In [ ]:
employees_df = spark.read.csv(DATA_PATH, header=True, inferSchema=True)
employees_df.printSchema()
print("Rows:", employees_df.count())   # 50

## Task 2: Define a schema for the data

**Spark maps a schema to the CSV by position, not by name.** `header=True` only skips the first line. `Salary` and `Age` are both integers, so swapping them would load with **no error** and silently corrupt every later result. The order below follows the file header: `Emp_No, Emp_Name, Salary, Age, Department`.

DataFrames are **immutable**: this doesn't "assign" a schema to the Task 1 DataFrame. It reads the file again and the variable now points to a new DataFrame.

In [ ]:
schema = StructType([
    StructField("Emp_No", IntegerType()),
    StructField("Emp_Name", StringType()),
    StructField("Salary", IntegerType()),
    StructField("Age", IntegerType()),
    StructField("Department", StringType()),
])

employees_df = spark.read.csv(DATA_PATH, header=True, schema=schema)
employees_df.show(5)   # check: Donald -> Salary 2600, Age 29 (matches the raw file)

## Task 3: Display the schema

Note `nullable = true` on every column: for CSV sources Spark does not enforce `nullable=False`, so it is not a data quality check.

In [ ]:
employees_df.printSchema()

## Task 4: Create a temporary view

SQL can't see Python variables; the view is a **name tag** in Spark's catalog. It copies no data. `createOrReplaceTempView` is idempotent (safe to re-run), while `createTempView` fails if the view already exists.

The view points to **this** DataFrame: if `employees_df` is reassigned later, the view must be re-created.

In [ ]:
employees_df.createOrReplaceTempView("employees")

## Task 5: SQL query - employees older than 30

In [ ]:
older_than_30 = spark.sql("""
    SELECT *
    FROM employees
    WHERE Age > 30
""")
older_than_30.show()
print("SQL count:      ", older_than_30.count())                      # 37

# Cross-check with the DataFrame API (same Catalyst plan underneath)
print("DataFrame count:", employees_df.filter(F.col("Age") > 30).count())   # 37

## Task 6: Average salary by department

Count, min and max are added as a **sanity check**: every average must lie between its min and max, and the counts must add up to 50. Without `ORDER BY`, SQL guarantees no order.

In [ ]:
salary_stats = spark.sql("""
    SELECT Department,
           ROUND(AVG(Salary), 2) AS Avg_Salary,
           COUNT(*)              AS Employee_Count,
           MIN(Salary)           AS Min_Salary,
           MAX(Salary)           AS Max_Salary
    FROM employees
    GROUP BY Department
    ORDER BY Avg_Salary DESC, Department ASC
""")
salary_stats.show()

> **Insight:** IT has the highest average (7,400), but one salary (24,000) pulls it up. IT's median is 5,400. Averages hide outliers; that's why count, min and max sit next to them.

## Task 7: Filter IT department employees (DataFrame API)

In [ ]:
it_employees = employees_df.filter(F.col("Department") == "IT")   # values are case-sensitive
it_employees.show()
print("IT employees:", it_employees.count())   # 10, matches Task 6

## Task 8: Add a 10% bonus

`1.10` cannot be represented exactly in binary floating point (IEEE 754), so `2600 * 1.10 = 2860.0000000000005`. Rounding fixes the display; real money columns belong in `DecimalType`.

`withColumn` returns a **new** DataFrame, saved as `bonus_df` so `employees_df` stays untouched.

In [ ]:
bonus_df = employees_df.withColumn("SalaryAfterBonus", F.round(F.col("Salary") * 1.10, 2))
bonus_df.printSchema()   # SalaryAfterBonus: double
bonus_df.show(5)

## Task 9: Maximum salary by age

In [ ]:
max_salary_by_age = (
    employees_df.groupBy("Age")
    .agg(F.max("Salary").alias("Max_Salary"))
    .orderBy(F.col("Age").desc())
)
max_salary_by_age.show()   # 14 age groups, 26 to 39

## Task 10: Self-join on `Emp_No`

DataFrame aliases tell the two sides apart. The result keeps **both** sides (10 columns), so an unqualified `select("Salary")` fails with `AMBIGUOUS_REFERENCE`.

Real-world use of a self-join: an employee/manager hierarchy (`e.Manager_No = m.Emp_No`).

In [ ]:
e1 = employees_df.alias("e1")
e2 = employees_df.alias("e2")
self_joined = e1.join(e2, F.col("e1.Emp_No") == F.col("e2.Emp_No"))
self_joined.show(5)
print("Rows:", self_joined.count(), "| Columns:", len(self_joined.columns))   # 50 | 10

# Duplicate keys would multiply rows: confirm Emp_No is unique
print("Unique Emp_No:", employees_df.select("Emp_No").distinct().count())    # 50

# Joining on the column NAME keeps a single key column
print("Columns with join on name:", len(e1.join(e2, "Emp_No").columns))       # 9

## Task 11: Average employee age

One number for the whole table: `agg()` without `groupBy`. The transformation is saved first and the action runs on its own line: `show()` returns `None`.

In [ ]:
avg_age = employees_df.agg(F.round(F.avg("Age"), 2).alias("Avg_Age"))
avg_age.show()   # 33.56

# Cross-check with SQL
spark.sql("SELECT ROUND(AVG(Age), 2) AS Avg_Age FROM employees").show()

## Task 12: Total salary by department

**Reconciliation:** each total should equal the Task 6 average x count, and the five totals should add up to the company total.

In [ ]:
total_salary = (
    employees_df.groupBy("Department")
    .agg(F.sum("Salary").alias("Total_Salary"))
    .orderBy(F.col("Total_Salary").desc())
)
total_salary.show()

employees_df.agg(F.sum("Salary").alias("Company_Total")).show()   # 309116

## Task 13: Sort by age (ascending) and salary (descending)

One sort, two keys. Directions belong to **columns** (`.asc()` / `.desc()`), not to the DataFrame.

In [ ]:
sorted_df = employees_df.orderBy(F.col("Age").asc(), F.col("Salary").desc())
sorted_df.show(50)

# Equivalent: employees_df.orderBy(["Age", "Salary"], ascending=[True, False])

## Task 14: Count employees in each department

`count("*")` counts rows (people); `count(column)` skips nulls. Finance and IT tie at 10, so `Department` is added as a tiebreaker to make the order deterministic.

In [ ]:
employees_per_department = (
    employees_df.groupBy("Department")
    .agg(F.count("*").alias("Employee_Count"))
    .orderBy(F.col("Employee_Count").desc(), F.col("Department").asc())
)
employees_per_department.show()   # 13 + 10 + 10 + 9 + 8 = 50, matches Task 6

## Task 15: Names containing the letter "o"

Lowercasing first makes the filter **case-insensitive**: no name starts with "O" today, but the filter stays correct if one is added.

In [ ]:
names_with_o = employees_df.filter(F.lower(F.col("Emp_Name")).contains("o"))
names_with_o.show()
print("DataFrame API:", names_with_o.count())   # 8

# Cross-check with SQL LIKE
print("SQL LIKE:     ", spark.sql("SELECT * FROM employees WHERE Emp_Name LIKE '%o%'").count())   # 8

## Summary

| Task | Result |
|---|---|
| Rows loaded | 50 |
| Older than 30 | 37 |
| Highest average salary | IT, 7,400 (median 5,400) |
| IT employees | 10 |
| Donald after bonus | 2,860.0 |
| Age groups / max at 39 | 14 / 24,000 |
| Self-join | 50 rows, 10 columns |
| Average age | 33.56 |
| Company total salary | 309,116 |
| Names with "o" | 8 |

Every result was confirmed by a second method or reconciled against another task.

In [ ]:
spark.stop()